# <font color="#418FDE" size="6.5" uppercase>**Bilddaten vorbereiten**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Laden, speichern und konvertieren kleine Bilder mit Pillow, OpenCV und NumPy. 
- Untersuchen Farbräume, Pixelwerte, Histogramme und einfache Bildoperationen. 
- Erstellen eine einheitliche Vorverarbeitung für kleine Bildordner oder synthetische Bilder. 


## **1. Bibliotheken und Bilder**

### **1.1. Versionen prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_01_01.jpg?v=1787633121" width="250">



>* Versionen vor der Bildarbeit prüfen
>* Gleiche Werkzeuge sichern reproduzierbare Ergebnisse

>* Bildbibliotheken treffen oft unsichtbare Annahmen
>* Versionen dokumentieren erleichtert Fehlersuche

>* Versionen stabilisieren Lernumgebungen und Fehlersuche
>* Dokumentierte Werkzeuge erleichtern präzise Problembeschreibung



In [ ]:
#@title Python-Code - Versionen prüfen

# Dieses Beispiel prüft wichtige Bibliotheksversionen.
# Versionen beeinflussen Bilddaten und Reproduzierbarkeit.
# Die Ausgabe zeigt eine kompakte Arbeitsgrundlage.

import numpy as np
import cv2
import PIL

# Die Versionswerte kommen direkt aus den installierten Paketen.
numpy_version = np.__version__
opencv_version = cv2.__version__
pillow_version = PIL.__version__

# Eine kleine synthetische RGB-Grafik prüft die Grundfunktion.
image_rgb = np.zeros((4, 4, 3), dtype=np.uint8)
image_rgb[:, :2] = [255, 0, 0]
image_rgb[:, 2:] = [0, 128, 255]

# OpenCV erwartet oft BGR statt RGB.
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
restored_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Diese Prüfung bestätigt die verlustfreie Kanalumordnung.
conversion_ok = np.array_equal(image_rgb, restored_rgb)
shape_text = str(image_rgb.shape)

print("NumPy-Version: " + numpy_version)
print("OpenCV-Version: " + opencv_version)
print("Pillow-Version: " + pillow_version)
print("Synthetisches RGB-Bild: Form " + shape_text)
print("RGB zu BGR und zurück korrekt: " + str(conversion_ok))



### **1.2. Bilder anzeigen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_01_02.jpg?v=1787633123" width="250">



>* Bildanzeige prüft Laden und sichtbare Fehler
>* Bibliotheken unterscheiden Kanäle, Datentypen und Werte

>* Bilder direkt im Notebook vergleichen
>* Darstellung zeigt Datenstruktur und Bibliotheksunterschiede

>* Farbkanäle können zwischen Bibliotheken vertauscht sein
>* Frühe Anzeige erkennt Konvertierungsfehler rechtzeitig



In [ ]:
#@title Python-Code - Bilder anzeigen

# Dieses Beispiel zeigt ein synthetisches Farbbild.
# Pillow, OpenCV und NumPy nutzen unterschiedliche Darstellungen.
# Die Anzeige macht Kanalreihenfolgen sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Wir erzeugen ein kleines RGB-Bild im Speicher.
height = 60
width = 90
rgb_array = np.zeros((height, width, 3), dtype=np.uint8)

# Drei Farbbereiche machen Kanalfehler leicht erkennbar.
rgb_array[:, :30] = [255, 0, 0]
rgb_array[:, 30:60] = [0, 255, 0]
rgb_array[:, 60:] = [0, 0, 255]

# Pillow kann direkt aus dem RGB-Array ein Bildobjekt bauen.
pillow_image = Image.fromarray(rgb_array, mode="RGB")
pillow_array = np.array(pillow_image)

# OpenCV erwartet bei Farbbildern häufig BGR statt RGB.
bgr_array = cv2.cvtColor(pillow_array, cv2.COLOR_RGB2BGR)
converted_rgb = cv2.cvtColor(bgr_array, cv2.COLOR_BGR2RGB)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
if converted_rgb.shape != (height, width, 3):
    raise ValueError("Die Bildform passt nicht zum erwarteten RGB-Bild.")

print("Synthetisches Bild: 60 x 90 Pixel, 3 Farbkanäle.")
print(f"NumPy-Form: {converted_rgb.shape}, Datentyp: {converted_rgb.dtype}.")
print(f"Pixel links oben als RGB: {converted_rgb[0, 0].tolist()}.")
print(f"Derselbe Pixel als OpenCV-BGR: {bgr_array[0, 0].tolist()}.")

# Matplotlib zeigt RGB-Arrays direkt als Farbbild an.
fig, ax = plt.subplots(figsize=(5, 3))
ax.imshow(converted_rgb)
ax.set_title("Anzeige eines synthetischen RGB-Bildes")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.3. Bildformate konvertieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_01_03.jpg?v=1787633125" width="250">



>* Bildformat passend zum Zweck wählen
>* Erhalt wichtiger Bildeigenschaften bewusst prüfen

>* Dateiformat und Speicherbild unterscheiden
>* Farbkanäle korrekt umwandeln

>* Verlust, Transparenz und Farbtiefe bewusst prüfen
>* Konvertierte Bilder visuell und technisch kontrollieren



In [ ]:
#@title Python-Code - Bildformate konvertieren

# Dieses Beispiel konvertiert ein synthetisches Farbbild.
# Pillow und OpenCV nutzen unterschiedliche Kanalreihenfolgen.
# Die Ausgabe zeigt Formateigenschaften und ein Kontrollbild.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Wir erzeugen ein kleines synthetisches RGB-Bild im Speicher.
height = 64
width = 64
x_values = np.linspace(0, 255, width, dtype=np.uint8)

red_channel = np.tile(x_values, (height, 1))
green_channel = np.flipud(red_channel)
blue_channel = np.full((height, width), 80, dtype=np.uint8)

rgb_array = np.stack((red_channel, green_channel, blue_channel), axis=2)
if rgb_array.shape != (height, width, 3):
    raise ValueError("Das RGB-Bild hat nicht die erwartete Form.")

# Pillow beschreibt das Bild mit einem Modus wie RGB.
pillow_image = Image.fromarray(rgb_array, mode="RGB")
grayscale_image = pillow_image.convert("L")
rgba_image = pillow_image.convert("RGBA")

# OpenCV erwartet für Farbbilder meist BGR statt RGB.
bgr_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)
roundtrip_rgb = cv2.cvtColor(bgr_array, cv2.COLOR_BGR2RGB)

# Diese Prüfung zeigt, ob die Kanalumwandlung verlustfrei war.
roundtrip_ok = np.array_equal(rgb_array, roundtrip_rgb)
alpha_values = np.array(rgba_image)[:, :, 3]
unique_alpha = np.unique(alpha_values)

print(f"Pillow-Modus vorher: {pillow_image.mode}")
print(f"Graustufen-Modus nach Konvertierung: {grayscale_image.mode}")
print(f"RGBA-Modus mit Alphakanal: {rgba_image.mode}")
print(f"OpenCV-Arrayform im Speicher: {bgr_array.shape}")
print(f"RGB nach BGR und zurück unverändert: {roundtrip_ok}")
print(f"Einzigartiger Alphawert im Beispiel: {int(unique_alpha[0])}")

# Wir zeigen das zurückkonvertierte RGB-Bild als Sichtprüfung.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(roundtrip_rgb)
ax.set_title("Synthetisches RGB-Bild nach BGR-Rückkonvertierung")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



## **2. Pixel und Farben**

### **2.1. RGB und BGR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_02_01.jpg?v=1787633116" width="250">



>* RGB-Pixel mischen Rot, Grün und Blau
>* Farbbilder haben Höhe, Breite und Kanäle

>* RGB und BGR unterscheiden die Kanalreihenfolge
>* Vertauschte Kanäle erzeugen falsche Farben

>* Vertauschte Kanäle verfälschen Analysen und Operationen
>* Farbräume prüfen, dokumentieren und visuell kontrollieren



In [ ]:
#@title Python-Code - RGB und BGR

# Dieses Beispiel zeigt RGB und BGR.
# Kanalreihenfolgen verändern den sichtbaren Farbeindruck.
# Die Grafik vergleicht korrekte und falsche Anzeige.

import numpy as np
import matplotlib.pyplot as plt

# Wir bauen ein kleines synthetisches RGB-Bild.
height = 60
width = 90
rgb_image = np.zeros((height, width, 3), dtype=np.uint8)

# Drei Farbbereiche machen die Kanäle gut erkennbar.
rgb_image[:, :30] = [255, 0, 0]
rgb_image[:, 30:60] = [0, 255, 0]
rgb_image[:, 60:] = [0, 0, 255]

# OpenCV würde dieselben Zahlen oft als BGR interpretieren.
bgr_like_image = rgb_image[:, :, ::-1].copy()
wrong_display = bgr_like_image
correct_display = bgr_like_image[:, :, ::-1]

# Eine einfache Prüfung schützt vor unerwarteten Bildformen.
if rgb_image.shape != (height, width, 3):
    raise ValueError("Das Bild muss Höhe, Breite und drei Kanäle haben.")

# Wir drucken wenige Pixelwerte zur Orientierung.
print("RGB-Pixel links:", rgb_image[0, 0].tolist())
print("BGR-Speicher links:", bgr_like_image[0, 0].tolist())
print("Falsche Anzeige macht Rot zu Blau.")
print("Korrekte Rückkonvertierung stellt Rot wieder her.")

# Ein einzelnes breites Bild zeigt beide Interpretationen nebeneinander.
separator = np.full((height, 6, 3), 255, dtype=np.uint8)
comparison_image = np.concatenate((wrong_display, separator, correct_display), axis=1)

# Die Achse zeigt Pixelpositionen im Vergleichsbild.
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(comparison_image)
ax.set_title("BGR falsch angezeigt links, korrektes RGB rechts")
ax.set_xlabel("Pixelposition in x-Richtung")
ax.set_ylabel("Pixelposition in y-Richtung")

# Textmarken erklären die beiden Bildhälften direkt.
ax.text(12, 8, "falsch", color="white", fontsize=12, weight="bold")
ax.text(108, 8, "korrekt", color="white", fontsize=12, weight="bold")
plt.show()



### **2.2. Graustufen und Transparenz**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_02_02.jpg?v=1787633118" width="250">



>* Graustufen zeigen Pixel als Helligkeitswerte
>* Umwandlung gewichtet Farben unterschiedlich

>* Alphakanal steuert die Sichtbarkeit von Pixeln
>* Transparenz bei Analysen immer berücksichtigen

>* Kanalstruktur vor Bildoperationen bewusst prüfen
>* Transparenz, Graustufen und Farbinformation gezielt behandeln



In [ ]:
#@title Python-Code - Graustufen und Transparenz

# Dieses Beispiel zeigt Graustufen und Transparenz.
# Pixelkanäle werden sichtbar und vergleichbar gemacht.
# Das Diagramm zeigt sichtbare Helligkeiten.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines synthetisches RGBA-Bild.
height = 8
width = 8
x_values = np.linspace(0, 255, width, dtype=np.uint8)

red_channel = np.tile(x_values, (height, 1))
green_channel = np.flipud(red_channel)
blue_channel = np.full((height, width), 80, dtype=np.uint8)
alpha_channel = np.tile(x_values, (height, 1))

rgba_image = np.stack(
    [red_channel, green_channel, blue_channel, alpha_channel], axis=2
)

# Diese Prüfung macht die Kanalstruktur bewusst.
if rgba_image.shape != (height, width, 4):
    raise ValueError("Das Bild muss vier Kanäle besitzen.")

# Graustufen gewichten Farben ähnlich der menschlichen Wahrnehmung.
gray_float = (
    0.299 * rgba_image[:, :, 0]
    + 0.587 * rgba_image[:, :, 1]
    + 0.114 * rgba_image[:, :, 2]
)

gray_image = np.clip(np.round(gray_float), 0, 255).astype(np.uint8)
alpha_float = rgba_image[:, :, 3].astype(np.float32) / 255.0

# Transparenz bestimmt, wie stark die Helligkeit sichtbar wird.
visible_gray = np.round(gray_image.astype(np.float32) * alpha_float)
visible_gray = np.clip(visible_gray, 0, 255).astype(np.uint8)

print("RGBA-Form:", rgba_image.shape)
print("Graustufen-Form:", gray_image.shape)
print("Alpha-Werte oben:", alpha_channel[0, :].tolist())
print("Graustufen oben:", gray_image[0, :].tolist())
print("Sichtbare Helligkeit oben:", visible_gray[0, :].tolist())

# Ein Histogramm zeigt nur die sichtbaren Helligkeiten.
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(visible_gray.ravel(), bins=8, range=(0, 255), color="gray", edgecolor="black")
ax.set_title("Histogramm der sichtbaren Graustufen")
ax.set_xlabel("sichtbare Helligkeit")
ax.set_ylabel("Pixelanzahl")
plt.show()



### **2.3. Pixelwerte skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_02_03.jpg?v=1787633120" width="250">



>* Pixelwerte in andere Zahlenbereiche umrechnen
>* Normierung erleichtert stabile Bildberechnungen

>* Normierung ändert meist nur Zahlenmaßstäbe.
>* Min-Max-Skalierung kann Kontrast und Rauschen verstärken.

>* Datentyp und Wertebereich immer prüfen
>* Einheitlich skalieren und Schritte dokumentieren



In [ ]:
#@title Python-Code - Pixelwerte skalieren

# Dieses Beispiel skaliert synthetische Pixelwerte.
# Es vergleicht Normierung und Kontraststreckung.
# Die Ausgabe zeigt veränderte Zahlenbereiche.

import numpy as np
import matplotlib.pyplot as plt

# Ein kleines Graubild macht die Zahlen gut sichtbar.
image_uint8 = np.array(
    [[40, 50, 60, 70], [80, 90, 100, 110], [120, 130, 140, 150]],
    dtype=np.uint8,
)

# Diese Prüfung schützt vor unerwarteten Bildformen.
if image_uint8.ndim != 2:
    raise ValueError("Erwartet wird ein zweidimensionales Graubild.")

# Normierung teilt feste Acht-Bit-Werte durch 255.
image_normalized = image_uint8.astype(np.float32) / 255.0

# Kontraststreckung nutzt Minimum und Maximum dieses Bildes.
pixel_min = float(image_uint8.min())
pixel_max = float(image_uint8.max())
pixel_range = pixel_max - pixel_min

# Diese Prüfung verhindert eine Division durch null.
if pixel_range == 0:
    raise ValueError("Alle Pixel sind gleich hell.")

# Jetzt wird der vorhandene Bereich auf null bis eins gestreckt.
image_stretched = (image_uint8.astype(np.float32) - pixel_min) / pixel_range

# Kurze Zahlen zeigen den Unterschied beider Skalierungen.
print("Originalbereich: 40 bis 150 als uint8")
print("Normiert: min", round(float(image_normalized.min()), 3), "max", round(float(image_normalized.max()), 3))
print("Gestreckt: min", round(float(image_stretched.min()), 3), "max", round(float(image_stretched.max()), 3))
print("Beispielpixel 90 normiert:", round(float(image_normalized[1, 1]), 3))
print("Beispielpixel 90 gestreckt:", round(float(image_stretched[1, 1]), 3))

# Ein Histogramm macht die verschobenen Wertebereiche sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(image_normalized.ravel(), bins=8, alpha=0.7, label="Normiert")
ax.hist(image_stretched.ravel(), bins=8, alpha=0.7, label="Gestreckt")
ax.set_title("Pixelwerte nach zwei Skalierungen")
ax.set_xlabel("Pixelwert im Bereich 0 bis 1")

ax.set_ylabel("Anzahl der Pixel")
ax.legend()
plt.show()



## **3. Bildoperationen**

### **3.1. Zuschneiden und Skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_03_01.jpg?v=1787633127" width="250">



>* Relevante Bildbereiche gezielt auswählen
>* Einheitliche Bildgrößen für weitere Verarbeitung

>* Ausschnitt passend zum Bildinhalt wählen
>* Ergebnisse visuell prüfen und dokumentieren

>* Skalieren verändert Details und Bildschärfe
>* Seitenverhältnis erhalten, Strategie dokumentieren



In [ ]:
#@title Python-Code - Zuschneiden und Skalieren

# Dieses Beispiel zeigt Zuschneiden und Skalieren.
# Ein synthetisches Bild bleibt vollständig im Speicher.
# Das Ergebnis vergleicht Original und Vorverarbeitung.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Wir erzeugen ein kleines synthetisches RGB-Bild.
height = 80
width = 120
image = np.full((height, width, 3), 235, dtype=np.uint8)

# Ein farbiges Objekt liegt bewusst in der Bildmitte.
image[20:60, 35:85, 0] = 40
image[20:60, 35:85, 1] = 120
image[20:60, 35:85, 2] = 220

# Diese Prüfung schützt vor falschen Bildformen.
if image.shape != (80, 120, 3):
    raise ValueError("Das Bild muss Höhe, Breite und drei Farbkanäle haben.")

# Wir schneiden einen zentralen Bereich aus.
crop_top = 10
crop_bottom = 70
crop_left = 25
crop_right = 95

# Der Zuschnitt enthält weniger Hintergrund.
cropped = image[crop_top:crop_bottom, crop_left:crop_right].copy()

# Pillow skaliert den Zuschnitt auf eine einheitliche Zielgröße.
target_size = (64, 64)
pil_cropped = Image.fromarray(cropped)
resized = np.array(pil_cropped.resize(target_size, Image.Resampling.BILINEAR))

# Wir legen Original und Ergebnis nebeneinander in ein Bild.
canvas = np.full((80, 200, 3), 255, dtype=np.uint8)
canvas[0:80, 0:120] = image
canvas[8:72, 136:200] = resized

print(f"Originalgröße: {image.shape[1]} x {image.shape[0]} Pixel")
print(f"Zuschnittgröße: {cropped.shape[1]} x {cropped.shape[0]} Pixel")
print(f"Zielgröße: {resized.shape[1]} x {resized.shape[0]} Pixel")
print("Links: Original, rechts: zugeschnitten und skaliert.")

# Eine einzige Achse zeigt den direkten visuellen Vergleich.
fig, ax = plt.subplots(figsize=(7, 3))
ax.imshow(canvas)
ax.set_title("Zuschneiden und Skalieren eines synthetischen Bildes")

ax.set_xlabel("Pixelposition in der Vergleichsansicht")
ax.set_ylabel("Pixelposition in der Vergleichsansicht")
ax.set_xticks([])
ax.set_yticks([])

plt.show()



### **3.2. Filter und Kanten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_03_02.jpg?v=1787633129" width="250">



>* Filter machen Bilder vergleichbarer und robuster
>* Glättung und Details bewusst ausbalancieren

>* Kanten zeigen wichtige Übergänge und Objektgrenzen.
>* Vorverarbeitung beeinflusst Kanten besonders bei kleinen Bildern.

>* Feste Pipeline mit gleichen Parametern nutzen
>* Filterwirkung prüfen, dann Einstellungen festlegen



In [ ]:
#@title Python-Code - Filter und Kanten

# Dieses Beispiel zeigt Filter und Kanten.
# Ein synthetisches Bild macht die Wirkung sichtbar.
# Die Pipeline verarbeitet alle Pixel gleich.

import numpy as np
import cv2
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines synthetisches Graustufenbild.
image = np.full((96, 96), 35, dtype=np.uint8)
image[24:72, 28:68] = 190
image[38:58, 42:82] = 115

# Ein festes Rauschmuster simuliert unruhige Aufnahmen.
rng = np.random.default_rng(42)
noise = rng.normal(0, 18, image.shape)
noisy_image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

# Die Vorverarbeitung nutzt feste Parameter für jedes Bild.
blurred_image = cv2.GaussianBlur(noisy_image, (5, 5), 0)
edges = cv2.Canny(blurred_image, 60, 140)

# Eine einfache Prüfung schützt vor unerwarteten Bildformen.
if noisy_image.shape != edges.shape:
    raise ValueError("Bild und Kantenbild müssen gleich groß sein.")

# Wir messen, wie viele Pixel als Kante erkannt wurden.
edge_share = 100 * np.count_nonzero(edges) / edges.size
mean_before = float(noisy_image.mean())
mean_after = float(blurred_image.mean())

print("Synthetisches Bild: 96 x 96 Pixel.")
print(f"Mittlere Helligkeit vor Filterung: {mean_before:.1f}.")
print(f"Mittlere Helligkeit nach Filterung: {mean_after:.1f}.")
print(f"Anteil erkannter Kantenpixel: {edge_share:.1f} %.")

# Für eine Achse kombinieren wir Original, Filter und Kanten nebeneinander.
combined = np.concatenate([noisy_image, blurred_image, edges], axis=1)

fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(combined, cmap="gray", vmin=0, vmax=255)
ax.set_title("Links: verrauscht, Mitte: geglättet, Rechts: Kanten")
ax.set_xlabel("Pixelposition in der kombinierten Ansicht")
ax.set_ylabel("Pixelzeile")
ax.set_xticks([48, 144, 240])
ax.set_xticklabels(["verrauscht", "geglättet", "Kanten"])
ax.set_yticks([0, 48, 95])
plt.show()



### **3.3. Bildordner vorbereiten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_05/Lecture_A/image_03_03.jpg?v=1787633130" width="250">



>* Bildordner einheitlich prüfen und vorbereiten
>* Klare Regeln ermöglichen verlässliche Analysen

>* Einheitliche Größe und Zuschnitte festlegen
>* Farbraum und Pixelwerte reproduzierbar normalisieren

>* Verarbeitung prüfen, dokumentieren und Fehler protokollieren
>* Rohdaten getrennt halten, Datensätze konsistent speichern



In [ ]:
#@title Python-Code - Bildordner vorbereiten

# Wir bereiten einen kleinen synthetischen Bildordner vor.
# Einheitliche Regeln machen Bilddaten vergleichbar und reproduzierbar.
# Am Ende sehen wir normalisierte Beispielbilder.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Diese Liste ersetzt hier einen kleinen Bildordner im Speicher.
rng = np.random.default_rng(42)
raw_images = []

# Drei synthetische Bilder haben unterschiedliche Größen und Kanäle.
raw_images.append(rng.integers(0, 256, size=(40, 60, 3), dtype=np.uint8))
raw_images.append(rng.integers(0, 256, size=(70, 45), dtype=np.uint8))
raw_images.append(rng.integers(0, 256, size=(55, 55, 4), dtype=np.uint8))

# Diese Funktion wendet dieselben Regeln auf jedes Bild an.
def preprocess_image(image_array, target_size=(32, 32)):
    if image_array.ndim == 2:
        pil_image = Image.fromarray(image_array, mode="L").convert("RGB")
    else:
        pil_image = Image.fromarray(image_array).convert("RGB")

    resized_image = pil_image.resize(target_size, Image.Resampling.BILINEAR)
    prepared_array = np.asarray(resized_image, dtype=np.float32) / 255.0
    return prepared_array

# Alle Bilder werden gleich konvertiert, skaliert und normalisiert.
prepared_images = []
for image_array in raw_images:
    prepared_images.append(preprocess_image(image_array))

prepared_batch = np.stack(prepared_images)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
expected_shape = (3, 32, 32, 3)
if prepared_batch.shape != expected_shape:
    raise ValueError("Die vorbereiteten Bilder haben eine unerwartete Form.")

# Kurze Kennzahlen zeigen, ob die Vorbereitung plausibel ist.
print("Anzahl vorbereiteter Bilder:", prepared_batch.shape[0])
print("Einheitliche Form:", prepared_batch.shape[1:])
print("Pixelbereich:", round(float(prepared_batch.min()), 3), "bis", round(float(prepared_batch.max()), 3))
print("Mittlere Helligkeit:", round(float(prepared_batch.mean()), 3))

# Für die Anzeige legen wir die Bilder nebeneinander in ein Raster.
preview_image = np.concatenate(prepared_batch, axis=1)

# Die Abbildung zeigt die einheitlich vorbereiteten Bilder.
fig, ax = plt.subplots(figsize=(7, 3))
ax.imshow(preview_image)
ax.set_title("Synthetische Bilder nach einheitlicher Vorverarbeitung")
ax.set_xlabel("Pixelposition im Vorschauraster")
ax.set_ylabel("Pixelposition")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Bilddaten vorbereiten**</font>


In this lecture, you learned to:
- Laden, speichern und konvertieren kleine Bilder mit Pillow, OpenCV und NumPy. 
- Untersuchen Farbräume, Pixelwerte, Histogramme und einfache Bildoperationen. 
- Erstellen eine einheitliche Vorverarbeitung für kleine Bildordner oder synthetische Bilder. 

In the next Lecture (Lecture B), we will go over 'Signale vorbereiten'